# Extracting Data for Country by ISO3

The Fathom flood data are stored as individual tiles, organized into folders per model on the GOST AWS bucket. We have generated virtual rasters (.vrt) for each model, making reading and extracting easier.

In [ ]:
import sys
import os
import rasterio

import geopandas as gpd

sys.path.insert(0, "../../src")

import GOSTrocks.dataMisc as dMisc
import GOSTrocks.rasterMisc as rMisc
from GOSTrocks.misc import tPrint

In [ ]:
os.path.abspath("../../src")

In [ ]:
iso3 = "KEN"
out_folder = f"/home/wb411133/temp/FATHOM/{iso3}"
if not os.path.exists(out_folder):
    os.makedirs(out_folder)

# This demo uses the default national boundaries included with GeoPandas, but this can be changed here
world_filepath = gpd.datasets.get_path("naturalearth_lowres")
world = gpd.read_file(world_filepath)
inB = world.loc[world["iso_a3"] == iso3].copy()

In [ ]:
# Select layer to download
flood_type = ["COASTAL", "FLUVIAL", "PLUVIAL"]
defence = ["DEFENDED"]
return_period = ["1in5", "1in10", "1in50"]
climate_model = ["PERCENTILE50"]
year = ["2020"]

# all_vrts is a pandas dataframe with all the vrt paths to the global datasets, with columns defining
# the various models' defining attributes
all_vrts = dMisc.get_fathom_vrts(True)
sel_images = all_vrts.loc[
    (all_vrts["FLOOD_TYPE"].isin(flood_type))
    & (all_vrts["DEFENCE"].isin(defence))
    & (all_vrts["RETURN"].isin(return_period))
    & (all_vrts["CLIMATE_MODEL"].isin(climate_model))
]

In [ ]:
all_vrts.head()

In [ ]:
sel_images.head()

In [ ]:
# For each image in the selected images dataframe, we clip out the area of interest
#     which is defined by the ioso3 code, but could be any GeoDataFrame

for idx, row in sel_images.iterrows():
    out_file = os.path.join(out_folder, os.path.basename(row["PATH"]))
    if not os.path.exists(out_file):
        cur_r = rasterio.open(row["PATH"])
        rMisc.clipRaster(cur_r, inB, out_file)
    tPrint(os.path.basename(row["PATH"]))